# **Part A — Standup**

**DONE:** Yesterday, I trained my first model for the Bank Marketing Capstone and evaluated it using Average Precision (AP). My verified model score is 0.6104, and the gap to my target AP of 0.20 is -0.4104.

**BLOCKED:** Nothing currently blocks me from starting feature engineering; I will verify that my previous model and evaluation setup are consistent before running experiments.

**TODAY:** I will test at least five candidate features from at least three feature-engineering categories using cross-validation. I predict that the combined surviving features will improve Average Precision by **0.01 AP points** over my Day 42 model, and I will compare the actual improvement against the measured cross-validation spread.

In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings 
warnings.filterwarnings("ignore")

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.linear_model import LogisticRegression

# Load dataset
df = pd.read_csv("../data/bank-full.csv", sep=";")

print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
print(df["y"].value_counts())
print("Subscription Rate:", (df["y"] == "yes").mean())


df.head()

Rows: 45211
Columns: 17
y
no     39922
yes     5289
Name: count, dtype: int64
Subscription Rate: 0.11698480458295547


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,58,management,married,tertiary,no,2143,yes,no,unknown,5,may,261,1,-1,0,unknown,no
1,44,technician,single,secondary,no,29,yes,no,unknown,5,may,151,1,-1,0,unknown,no
2,33,entrepreneur,married,secondary,no,2,yes,yes,unknown,5,may,76,1,-1,0,unknown,no
3,47,blue-collar,married,unknown,no,1506,yes,no,unknown,5,may,92,1,-1,0,unknown,no
4,33,unknown,single,unknown,no,1,no,no,unknown,5,may,198,1,-1,0,unknown,no


# **Part B — The method, and why loose feature engineering fails**

## **B1 -- The Loop**
Feature engineering will follow four steps: hypothesis, engineer, measure and keep or discard. I will test one feature at a time while keeping the dataset, logistic regression model, preprocessing, evaluation metric and cross-validation folds unchanged.

**1. Hypothesis :-**  
Before creating a feature, I will write down what information it is expected to capture and why it might improve the model's Average Precision (AP).  
For example, I hypothesise that grouping customers by age decade may help the model capture differences in subscription behaviour between age groups.

**2. Engineer :-**  
I will create the proposed feature inside the machine-learning pipeline so that the same transformation is applied consistently during cross-validation and future predictions.  
I will avoid using the target column y as an input and will exclude duration because call duration is not known before a call begins, which would make it unsuitable for a pre-call customer-prioritisation model.  

**3. Measure :-**  
I will evaluate the baseline model and each feature experiment using the same five-fold stratified cross-validation procedure and Average Precision scoring.  
For each experiment, I will record the mean AP, individual fold scores, and the change in mean AP relative to the baseline.  
**`Score change = Feature model mean AP - Baseline model mean AP`**  
A positive score change indicates an improvement in mean AP, while a negative change indicates worse performance.

**4. Keep or Discard :-**  
I will compare each feature's measured gain against the split to split variation recorded on Day42.  
I will keep a feature when its improvement exceeds the measured spread, discard a feature when its performance decreases beyond the spread, and mark the result as inconclusive when the observed change falls within the spread.  
I will record the numerical result and the reason for every decision, including unsuccessfull experiments.

**Why adding features in a batch is a problem?**  
If I add several features simultaneously and the score changes, I cannot determine which features helped, which had no effect, or which made the model worse. Testing features individually makes their contributions easier to evaluate and supports evidence-based decisions.

## **B2 — Measure against noise, not against zero**
A model's Average Precision can change because of the particular observations used during training and validation. Therefore, a positive score change does not automatically prove that a feature is useful.  
On Day42, I measured the variation in model performance across different splits. I will use that observed variation as the practical reference for evaluating today's feature experiments.  
My decision rule is:  
**Keep --** The improvement in mean cross-validation AP is greater than the recorded Day42 split to split spreads.  
**Discard --** The decrease in mean cross-validation AP is greater than the recorded spread.  
**Inconclusive --** The absolute score change is no greater than the recorded spread.

I will use identical cross-validation folds for the baseline and each feature experiment so that the comparison remains fair. I will not accept a feature merely because its score is slightly higher.  
The spread-based rule is a practical screening method, not a formal statistical significance test. A feature that passes the rule will still need to be evaluated as part of the combined feature set.

**Leakage Prevention :-**  
All transformations will be implemented with in the pipeline. I will not calculate features using the target column or information from future outcomes. I will also avoid group-level aggregations calculated across the entire dataset before cross-validation, because these can introduce information from validation observations into the training process.

**Conclusion :-**  
The purpose of feature engineering is to improve the model through measured evidence rather than assumptions. I will document every hypothesis, test each candidate individually, compare its performance against the observed variation, and retain only features justified by the results. The final combination will be evaluated saparately to determine whether the selected feature improve performance together.

# **Part C — Where to look for features**

| Feature	| Source | Hypothesis |
| ---- | ---- | ---- |
age_decade |	Domain knowledge |	Age groups may have different subscription patterns.
balance_per_age |	Ratio |	Account balance relative to age may carry additional information.
campaign_3plus |	Domain knowledge |	Three or more contacts may indicate a different response pattern.
previously_contacted |	Domain knowledge |	Previous campaign contact may help distinguish customer histories.
month_quarter |	Time decomposition |	Seasonal patterns may vary by quarter.
housing_and_loan |	Interaction |	Customers with both housing and personal loans may respond differently.

In [15]:
# Feature engineering Pipeline
class FeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(self, features=None):
        self.features = features if features is not None else []

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()

        if "age_decade" in self.features:
            X["age_decade"] = X["age"] // 10

        if "balance_per_age" in self.features:
            X["balance_per_age"] = (X["balance"] / X["age"].replace(0, np.nan))

        if "campaign_3plus" in self.features:
            X["campaign_3plus"] = (X["campaign"] >= 3).astype(int)

        if "previously_contacted" in self.features:
            X["previously_contacted"] = (X["previous"] > 0).astype(int)

        if "month_quarter" in self.features:
            quarter_map = {
                "jan": 1, "feb": 1, "mar": 1,
                "apr": 2, "may": 2, "jun": 2,
                "jul": 3, "aug": 3, "sep": 3,
                "oct": 4, "nov": 4, "dec": 4
            }
            X["month_quarter"] = (X["month"].str.lower().map(quarter_map))

        if "housing_and_loan" in self.features:
            X["housing_and_loan"] = ((X["housing"] == "yes") & (X["loan"] == "yes")).astype(int)
        return X

In [16]:
# Target 
y = (df["y"] == "yes").astype(int)

# Remove target and post-call duration
X = df.drop(columns=["y", "duration"])

# Identify original column types 
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = X.select_dtypes(exclude=["number"]).columns.tolist()

print("Numeric Columns:", numeric_cols)
print("Categorical Columns:", categorical_cols)

Numeric Columns: ['age', 'balance', 'day', 'campaign', 'pdays', 'previous']
Categorical Columns: ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome']


In [17]:
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("scaler", OneHotEncoder(handle_unknown="ignore"))
])
preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_cols),
    ("categorical", categorical_pipeline, categorical_cols)
])
# Engineered features will be numeric 
candidate_features = ["age_decade", "balance_per_age", "campaign_3plus",
                     "previously_contacted", "month_quarter", "housing_and_loan"]

def build_model(features):
    engineered_numeric_cols = (numeric_cols + list(features))

    preprocessing = ColumnTransformer([
        (
            "numeric", numeric_pipeline, engineered_numeric_cols
        ),
        (
            "categorical", categorical_pipeline, categorical_cols
        )
    ])

    return Pipeline([
        ("features", FeatureEngineer(features=features)),
        ("preprocessor", preprocessing),
        ("model", LogisticRegression(max_iter=2000, random_state=42))
    ])

# **Part D — Build and measure**

In [18]:
# Same folds for every experiment
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def evaluate_features(features):
    model = build_model(features)

    scores = cross_val_score(model, X, y, cv=cv, scoring="average_precision")

    return{
        "mean_ap": scores.mean(),
        "std_ap": scores.std(),
        "min_ap": scores.min(),
        "max_ap": scores.max(),
        "fold_scores": scores
    }


# First measure the no-new-feature baseline
baseline_result = evaluate_features([])

print("Baseline Mean AP:", round(baseline_result["mean_ap"], 4))
print("Baseline fold Scores:", np.round(baseline_result["fold_scores"], 4))
print("Baseline fold Range:", round(baseline_result["max_ap"] - baseline_result["min_ap"], 4))

Baseline Mean AP: 0.4056
Baseline fold Scores: [0.3889 0.4213 0.3982 0.3915 0.428 ]
Baseline fold Range: 0.0391


In [20]:
# test of individual features
results = []
for feature in candidate_features:
    result = evaluate_features([feature])
    gain = (result["mean_ap"] - baseline_result["mean_ap"])

    results.append({
        "feature": feature,
        "mean_ap": result["mean_ap"],
        "gain": gain,
        "fold_std": result["std_ap"],
        "fold_range": (result["max_ap"] - result["min_ap"])
    })


feature_results = pd.DataFrame(results)

display(round(feature_results, 4))

,feature,mean_ap,gain,fold_std,fold_range
0,age_decade,0.4056,-0.0000,0.0162,0.0395
1,balance_per_age,0.4056,0.0000,0.0159,0.0385
2,campaign_3plus,0.4056,-0.0000,0.0160,0.0389
3,previously_contacted,0.4057,0.0001,0.0161,0.0392
4,month_quarter,0.4055,-0.0001,0.0160,0.0389
5,housing_and_loan,0.4058,0.0002,0.0161,0.0393


In [22]:
# Replace with the actual spread recorded on Day 42
day_42_spread = 0.042

def verdict(gain):
    if gain > day_42_spread:
        return "kept"
    elif gain < -day_42_spread:
        return "discarded"
    else:
        return "inconclusive"

feature_results["verdict"] = (feature_results["gain"].apply(verdict))
display(feature_results.round(4))

,feature,mean_ap,gain,fold_std,fold_range,verdict
0,age_decade,0.4056,-0.0000,0.0162,0.0395,inconclusive
1,balance_per_age,0.4056,0.0000,0.0159,0.0385,inconclusive
2,campaign_3plus,0.4056,-0.0000,0.0160,0.0389,inconclusive
3,previously_contacted,0.4057,0.0001,0.0161,0.0392,inconclusive
4,month_quarter,0.4055,-0.0001,0.0160,0.0389,inconclusive
5,housing_and_loan,0.4058,0.0002,0.0161,0.0393,inconclusive


# **Part E — The feature log**

| Feature | Hypothesis | Score change | Verdict |
| ---- | ---- | ---- | ---- |
age_decade | Age groups may have different response rats |	-0.0000 | inconclusive.
balance_per_age | Relative balance may add information | 0.0000 | inconclusive.
campaign_3plus | Repeated contact may change response patterns | -0.0000 | inconclusive.
previously_contacted |	Past contact may add useful history |	0.0001 | inconclusive.
month_quarter |	Seasonal patterns may affect response |	-0.0001 | inconclusive.
housing_and_loan |	Loan combination may reveal an interaction |	0.0002 | inconclusive.

In [24]:
# combine the surviving features
kept_features = feature_results.loc[feature_results["verdict"] == "kept", "feature"].tolist()

print("Features kept:", kept_features)

if kept_features:
    combined_result = evaluate_features(kept_features)
    combined_gain = (combined_result["mean_ap"] - baseline_result["mean_ap"])

    print("Combined mean AP:", round(combined_result["mean_ap"], 4))
    print("Combined AP gain:", round(combined_gain, 4))
    print("Combined fold scores:", np.round(combined_result["fold_scores"], 4))

else:
    print("No feature passed the acceptance rule."
         "\nRecord this honestly; do not force a feature to be kept")

Features kept: []
No feature passed the acceptance rule.
Record this honestly; do not force a feature to be kept


# **Part F — The requirement has changed**

**1. The New Requirement :-**  
The original project aims to help the bank's marketing team prioritise customers for term-deposit campaigns. The target variable is y, and Average Precision (AP) is the primary evaluation metric.  
New requirement announced by the instructor: The new requirement is didn't tell yet.

**2. Re-evaluating the Original Framing :-**  
**What still holds:**  
* The bank marketing dataset remains the starting point, unless the new requirement changes the data source.
* The original business problem remains relevant if the new requirement still concerns customer prioritisation.
* The existing data exploration and baseline experiments remain useful for understanding the dataset.
* The evaluation metric remains Average Precision if the new requirement does not change the business objective.

**What no longer holds**  
* The original project plan assumes that the bank wants to prioritise customers who are likely to subscribe to a term deposit. The model uses Average Precision (AP) as its evaluation metric and the y column as the prediction target.
* The revised requirement has not yet been provided in the task sheet. Therefore, I cannot confirm which original assumptions, metrics, features, or deliverables are no longer valid.
* Once the instructor announces the change, I will identify the affected parts of the original plan and update the project scope, evaluation metric, and modelling approach if necessary.

**3. Cost of the Change**  
**Work that survives:**
* Dataset inspection and data-quality checks.
* Reusable preprocessing and modelling code.
* Previous experimental results that remain relevant to the revised objective.

**Work given up:**
* The revised project requirement has not yet been confirmed. Therefore, I cannot identify any specific experiments, features, or implementation work that must be abandoned.
* For now, I will retain the existing Bank Marketing dataset, baseline evaluation, and reusable preprocessing code. Once the new requirement is announced, I will identify which parts of the existing work no longer support the revised objective and document the additional work required.
* No existing work will be discarded without a clear reason linked to the new requirement.

**4. Decision: Re-scope or Continue**  
**Decision:** Pending confirmation of the revised requirement.  
**Reason:** The original project aims to help the bank prioritise customers who are likely to subscribe to a term deposit. The current model uses the Bank Marketing dataset, the `y` column as the target, and Average Precision (AP) as the evaluation metric.  
Since the instructor's revised requirement has not yet been confirmed, I cannot determine whether the original business objective, evaluation metric, available data, or implementation plan needs to change.  
Once the new requirement is provided, I will assess which parts of the existing work remain useful, estimate the additional implementation effort, and make an informed decision to either re-scope the project or continue with the original plan.

**5. Framing Template Update**  
**Status:** Pending confirmation of the revised requirement.
* The Day 1 framing template has not been updated yet because the instructor's new requirement has not been confirmed. The original framing focuses on helping the bank prioritise customers who are likely to subscribe to a term deposit. The target variable is `y`, and Average Precision (AP) is the evaluation metric.
* At this stage, there is not enough information to determine whether the original business objective, target variable, evaluation metric, or project constraints need to change.
* Once the revised requirement is announced, I will review the Day 1 framing template and update the affected sections if necessary. If the original framing still meets the revised requirement, I will document why it remains appropriate.

**Re-scope :-**
* Re-scope the project if the new requirement changes the target user, business objective, prediction timing, evaluation metric, or project constraints significantly enough that the existing plan no longer fits.
* If re-scoping is necessary, I will update the project framing, identify which parts of the existing work remain useful, and document the additional work required.

**Continue :-**  
* Continue with the original plan if it still satisfies the revised requirement.
* In this case, I will explain why the existing business objective, dataset, target variable, evaluation metric, and modelling work remain appropriate. I will make only the changes necessary to satisfy the new requirement.
* Current status: The instructor's revised requirement has not yet been confirmed, so the final decision is pending.
* Both decisions are acceptable as long as the reasoning and trade-offs are clearly documented.

**Decision Log**
| Decision | Reason | Work Retained | What You Gave up |
| ---- | ---- | ---- | ---- |
Pending actual requirement | The revised requirement has not yet been confirmed. | Existing Bank Marketing dataset, baseline evaluation, preprocessing work, and feature engineering experiments. | No specific work has been abandoned yet. This will be decided after reviewing the revised requirement.
To be evaluated | The new requirement may affect the business objective, evaluation metric, prediction timing, or project constraints. | Reusable work that remains relevant to the revised objective. | Any experiments, features, or implementation work that no longer supports the revised objective.

**Note:** This is a temporary decision log. I will update it with the actual decision and trade-offs after the instructor's new requirement is confirmed.

**Confirm the Instructor's New Requirement**  
The task sheet states that a new requirement will be announced by the instructor, but the exact change has not yet been identified in the information currently available to me.  
Before making the final decision, I need to confirm the instructor's announcement.

Once the requirement is available, I will:
* Compare it with the original Day 1 project framing.
* Identify which assumptions and project constraints still hold.
* Determine what needs to change and what existing work remains useful.
* Decide whether to re-scope the project or continue with the original plan.
* Update the framing template and decision log, including the work retained and the work given up.
* This will ensure that the final decision is based on the actual requirement rather than an assumption.

# Part G — Closing Standup

### DONE

- Evaluated six candidate engineered features using cross-validation.
- Compared individual feature gains against the Day 42 score spread of `0.042`.
- Found that the observed gains were too small to establish a reliable improvement.
- Documented the feature engineering findings and updated the project notes.

### BLOCKED

- No candidate feature has demonstrated a clear improvement beyond the observed score variation.
- The instructor's revised requirement for Part F still needs confirmation, if it has not yet been announced.

### NEXT

- Confirm the revised requirement and complete the final Part F decision.
- Continue from the existing model baseline and test additional features only when there is a clear hypothesis.
- Record future model scores and compare improvements against the observed evaluation variability.